In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import sys
sys.path.append('../')
from CEmulator.Emulator import CBaseEmulator

Omegab = 0.055
Omegam = 0.353
H0     = 63.6
ns     = 0.964
sigma8 = 0.781
mnu    = 0.06  # One single massive neutrino with two massless neutrinos
w      = -0.42
wa     = -1.75
Omegan = mnu/93.14/H0**2 * 1e4
Omegacb  = round(Omegam-Omegan, 8)
Omegac   = Omegacb - Omegab
print(Omegacb, Omegac)

csstemu = CBaseEmulator(verbose=True)
zlists  = np.array([0.0, 1.0, 2.0, 3.0])
##### need extra package [CLASS/CAMB] to transform sigma8 to As
csstemu.set_cosmos(Omegab=Omegab, Omegac=Omegac, H0=H0, sigma8=sigma8, ns=ns, 
                   w=w, wa=wa, mnu=mnu, checkbound=False, sigma8type='CLASS')
As = csstemu.Cosmo.As
print(As)

# First condition: find weq or (w0,wa)eq

In [ ]:
%%time
#First, obtain the equivalent w0, wa for the original cosmology
from CEmulator.SpectralEquivalence import SpectralEquivalence_DE

w0_arr, wa_arr = SpectralEquivalence_DE(csstemu, zlists, wCDM=False)
print(w0_arr)
print(wa_arr)

In [ ]:
%%time
wconst_arr = SpectralEquivalence_DE(csstemu, zlists, wCDM=True)
print(wconst_arr)

In [ ]:
ztt = np.linspace(0, 10, 1000)
with plt.style.context('default'):
    plt.plot(ztt, (w+wa*ztt/(1+ztt)), label=r"$w_0=%.2f, w_a=%.2f$"%(w, wa))
    plt.plot(zlists, (w0_arr + wa_arr*zlists/(1+zlists)), label=r"$(w_0,\, w_a)_{eq}(z)$")
    plt.xlim(3, 0)
    plt.grid(True)
    plt.legend()
    plt.xlabel(r'$z$')
    plt.ylabel(r"$w(z)$")

### neutrino_matter_like option

The neutrino_matter_like option allows you to treat neutrinos as matter in the background evolution. This can accelerate the calculation of comoving distance, leading to shorter time of the finding of equivalent dark energy models [i.e., $w_{\rm eq}$ or $(w_{0}, w_{a})_{\rm eq}$].

This approximation should be used with caution, as it may not be accurate for all cosmological models, especially those with significant neutrino contributions. It is recommended to compare the results obtained with this option against more accurate methods to ensure the validity of the approximation in your specific case.

In [ ]:
from CEmulator.SpectralEquivalence import SpectralEquivalence_DE

In [ ]:
%%time
wconst_arr = SpectralEquivalence_DE(csstemu, [0.5], wCDM=True, method="finite_diff_newton")
print(wconst_arr)

In [ ]:
%%time
wconst_arr = SpectralEquivalence_DE(csstemu, [0.5], wCDM=True, method="newton", neutrino_matter_like=False)
print(wconst_arr)

In [ ]:
%time
wconst_arr = SpectralEquivalence_DE(csstemu, [0.5], wCDM=True, method="newton", neutrino_matter_like=True)
print(wconst_arr)

In [ ]:
%%time
w0_arr, wa_arr = SpectralEquivalence_DE(csstemu, [0.5], wCDM=False, neutrino_matter_like=False)
print(w0_arr, wa_arr)

In [ ]:
%%time
w0_arr, wa_arr = SpectralEquivalence_DE(csstemu, [0.5], wCDM=False, neutrino_matter_like=True)
print(w0_arr, wa_arr)

# Second condition: find As or sigma8

In [ ]:
%%time
# Then, obtain the equivalent As for the original cosmology
from CEmulator.SpectralEquivalence import SpectralEquivalence_As

Aseq_arr = SpectralEquivalence_As(csstemu, zlists, w0_arr, wa_arr, sigma8_type='CLASS')
print(As * 1e9)
print(Aseq_arr * 1e9)

In [ ]:
%%time
# Then, obtain the equivalent As for the original cosmology
from CEmulator.SpectralEquivalence import SpectralEquivalence_As

Aseq_arr = SpectralEquivalence_As(csstemu, zlists, w0_arr, wa_arr, sigma8_type='CAMB')
print(As * 1e9)
print(Aseq_arr * 1e9)